# Поиск релевантных статей поддержки Авито

Для каждого запроса из `test.f` нужно вернуть до 10 подходящих `article_id`.  
В решении объединены два канала: модели по размеченным запросам из `calibration.f` и прямой поиск по полному корпусу статей.

Финальный результат на платформе: **MAP@10 = 0.610612**.

Ноутбук создаёт файл `answer.csv`. Файлы `articles.f`, `calibration.f` и `test.f` нужно положить в папку `candidate_data/` или рядом с ноутбуком.


## 1. Установка зависимостей

In [ ]:
%pip install -q \
    numpy==2.3.5 \
    pandas==2.2.3 \
    scipy==1.17.0 \
    scikit-learn==1.8.0 \
    pyarrow==25.0.0 \
    beautifulsoup4==4.14.3 \
    lxml==6.1.1 \
    pymorphy3==2.0.6 \
    pymorphy3-dicts-ru==2.4.417150.4580142

## 2. Настройки и загрузка данных

In [ ]:
from __future__ import annotations

import html
import os
import re
import warnings
from pathlib import Path

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(name, "1")

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from pymorphy3 import MorphAnalyzer
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Label not .* is present in all training examples")

TOP_K = 10
SEED = 42

UNKNOWN_ARTICLE_WEIGHT = 1.0
ARTICLE_CHANNEL_WEIGHT = 1.6
OUTPUT_FILE = Path("answer.csv")

np.random.seed(SEED)

In [ ]:
def find_data_dir() -> Path:
    candidates = [Path("candidate_data"), Path("."), Path("/content/candidate_data"), Path("/content")]
    required = {"articles.f", "calibration.f", "test.f"}

    for candidate in candidates:
        if candidate.exists() and required.issubset({path.name for path in candidate.iterdir()}):
            return candidate

    raise FileNotFoundError(
        "Не найдены articles.f, calibration.f и test.f. "
        "Положите их в candidate_data/ или рядом с ноутбуком."
    )


DATA_DIR = find_data_dir()
articles = pd.read_feather(DATA_DIR / "articles.f")
calibration = pd.read_feather(DATA_DIR / "calibration.f")
test = pd.read_feather(DATA_DIR / "test.f")

assert {"article_id", "title", "body"}.issubset(articles.columns)
assert {"query_id", "query_text", "ground_truth"}.issubset(calibration.columns)
assert {"query_id", "query_text"}.issubset(test.columns)

## 3. Обработка текста и HTML

Из HTML убираются `script`, `style` и `noscript`. Через `BeautifulSoup` извлекается основной текст, а отдельно - заголовки, подписи вкладок и спойлеров, названия столбцов таблиц и `alt` у изображений.

Текст приводится к нижнему регистру, ссылки и лишние символы удаляются, русские слова лемматизируются через `pymorphy3`. Часть обычных стоп-слов оставлена: например, `не`, `где`, `когда`, `до` и `после` для запросов поддержки реально меняют смысл.


In [ ]:
GREETING_RE = re.compile(
    r"\b(?:здравствуй(?:те)?|добрый\s+(?:день|вечер|утро)|"
    r"привет(?:ствую)?|подскажите(?:\s+пожалуйста)?|"
    r"скажите(?:\s+пожалуйста)?|пожалуйста|прошу|"
    r"хотел[аи]?\s+(?:бы\s+)?(?:узнать|уточнить)|"
    r"у\s+меня\s+(?:такой\s+)?вопрос|"
    r"можете\s+(?:ли\s+)?(?:подсказать|помочь)|помогите)\b",
    flags=re.IGNORECASE,
)
PLACEHOLDER_RE = re.compile(r"<(?:money|date|id|phone|url)>", flags=re.IGNORECASE)
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
TOKEN_RE = re.compile(r"[a-zа-я0-9]+")
RUSSIAN_TOKEN_RE = re.compile(r"[а-я]+")

STOPWORDS = set(
    "и в во не что он на я с со как а то все она так его но да ты к у же вы за "
    "бы по только ее мне было вот от меня еще нет о из ему теперь когда даже ну "
    "вдруг ли если уже или ни быть был него до вас нибудь опять уж вам ведь там "
    "потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была "
    "сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому "
    "этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были "
    "куда зачем сказать всех никогда сегодня можно при наконец два об другой хоть "
    "после над больше тот через эти нас про всего них какая много разве три эту моя "
    "впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более "
    "всегда конечно всю между это этот эта эти".split()
)

STOPWORDS -= {
    "не", "нет", "нельзя", "без",
    "до", "после", "через", "сейчас", "сегодня", "уже", "еще", "потом", "перед",
    "когда", "где", "куда", "как", "можно", "надо", "только",
}


class TextNormalizer:
    def __init__(self) -> None:
        self.morph = MorphAnalyzer()
        self.cache: dict[str, str] = {}

    def normalize(self, text: object, *, lemmatize: bool) -> str:
        value = html.unescape(str(text)).lower().replace("ё", "е")
        value = PLACEHOLDER_RE.sub(" placeholder ", value)
        value = URL_RE.sub(" ", value)
        tokens = [
            token for token in TOKEN_RE.findall(value)
            if token not in STOPWORDS and len(token) > 1
        ]

        if lemmatize:
            for i, token in enumerate(tokens):
                if len(token) > 2 and RUSSIAN_TOKEN_RE.fullmatch(token):
                    if token not in self.cache:
                        self.cache[token] = self.morph.parse(token)[0].normal_form
                    tokens[i] = self.cache[token]

        return " ".join(tokens)


def normalize_basic(text: object, *, remove_stopwords: bool) -> str:
    value = html.unescape(str(text)).lower().replace("ё", "е")
    value = PLACEHOLDER_RE.sub(" placeholder ", value)
    value = URL_RE.sub(" ", value)
    value = GREETING_RE.sub(" ", value)
    value = re.sub(r"[^0-9a-zа-я]+", " ", value)
    tokens = value.split()

    if remove_stopwords:
        tokens = [token for token in tokens if token not in STOPWORDS and len(token) > 1]

    return " ".join(tokens)

In [ ]:
def extract_article_text(raw_html: object) -> tuple[str, str]:
    soup = BeautifulSoup(str(raw_html), "lxml")

    for node in soup(["script", "style", "noscript"]):
        node.decompose()

    structure = []
    selectors = ["h1", "h2", "h3", "h4", "h5", "h6", ".tab-label", ".spoiler-title", "caption", "th"]

    for selector in selectors:
        for node in soup.select(selector):
            text = re.sub(r"\s+", " ", node.get_text(" ")).strip()
            if text:
                structure.append(text)

    for image in soup.find_all("img"):
        alt = re.sub(r"\s+", " ", str(image.get("alt", ""))).strip()
        if alt:
            structure.append(alt)

    body = re.sub(r"\s+", " ", soup.get_text(" ")).strip()
    return body, " ".join(dict.fromkeys(structure))

## 4. Ранжирование

Получилось два основных канала.

**По размеченным запросам**

* TF-IDF по словам и символам;
* `OneVsRestClassifier(LogisticRegression)`;
* kNN по похожим запросам;
* небольшой перенос оценки между статьями, которые часто встречались вместе.

**По статьям**

* BM25 по полному тексту;
* BM25 по перекрывающимся фрагментам статьи;
* TF-IDF по словам для полного текста;
* символьный TF-IDF по заголовкам и структуре HTML.

Параметры проверялись с помощью повторной пятикратной кросс-валидации: 5 запусков по 5 фолдов, метрика — MAP@10. Потом отдельно смотрел ошибки руками. Основные проблемы были такие: частые статьи занимали почти весь список из 10 результатов, отрицания терялись при очистке, длинные статьи плохо сравнивались с короткими вопросами, а статьи вне разметки `calibration.f` получали слишком маленький вес.

Поэтому были добавлены частотный штраф, сохранение смысловых слов, поиск по фрагментам и отдельный канал по всем статьям. Для финальной отправки вес статей, которых не было среди правильных ответов в `calibration.f`, был установлен равным `1.0`.

In [ ]:
def row_scale(scores: np.ndarray) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float32)
    scores = scores - scores.min(axis=1, keepdims=True)
    return scores / np.maximum(scores.max(axis=1, keepdims=True), 1e-8)


def blend(parts: list[tuple[np.ndarray, float]]) -> np.ndarray:
    result = None
    for scores, weight in parts:
        part = weight * row_scale(scores)
        result = part if result is None else row_scale(result) + part
    return row_scale(result)


def top_k_indices(row: np.ndarray, article_ids: np.ndarray, top_k: int = 10) -> np.ndarray:
    columns = np.arange(len(article_ids))
    candidates = np.argpartition(-row, top_k - 1)[:top_k]
    return candidates[np.lexsort((columns[candidates], -row[candidates]))]


def knn_scores(
    similarity: np.ndarray,
    y: np.ndarray,
    k: int,
    power: float,
    gamma: float,
) -> np.ndarray:
    k = min(k, len(y))
    indices = np.argpartition(-similarity, k - 1, axis=1)[:, :k]
    weights = np.take_along_axis(similarity, indices, axis=1)
    weights = np.maximum(weights, 0.0) ** power
    scores = np.einsum("nk,nkc->nc", weights, y[indices], optimize=True)

    frequencies = np.maximum(y.sum(axis=0), 1.0)
    return (scores / frequencies[None, :] ** gamma).astype(np.float32)

In [ ]:
def supervised_scores(
    train_queries: pd.DataFrame,
    eval_queries: pd.DataFrame,
    y: np.ndarray,
    normalizer: TextNormalizer,
) -> np.ndarray:
    all_queries = pd.concat(
        [train_queries["query_text"], eval_queries["query_text"]],
        ignore_index=True,
    ).astype(str).tolist()
    n_train = len(train_queries)

    compact = [normalize_basic(text, remove_stopwords=True) for text in all_queries]
    plain = [normalizer.normalize(text, lemmatize=False) for text in all_queries]
    lemma = [normalizer.normalize(text, lemmatize=True) for text in all_queries]

    compact_vectorizer = TfidfVectorizer(
        dtype=np.float32,
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=1,
    )
    compact_train = compact_vectorizer.fit_transform(compact[:n_train]).tocsr()
    compact_eval = compact_vectorizer.transform(compact[n_train:]).tocsr()
    similarity = (compact_eval @ compact_train.T).toarray().astype(np.float32)

    lemma_word_vectorizer = TfidfVectorizer(
        dtype=np.float32,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=1,
    )
    lemma_char_vectorizer = TfidfVectorizer(
        dtype=np.float32,
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=1,
    )

    word_train = lemma_word_vectorizer.fit_transform(lemma[:n_train]).tocsr()
    word_eval = lemma_word_vectorizer.transform(lemma[n_train:]).tocsr()
    char_train = lemma_char_vectorizer.fit_transform(plain[:n_train]).tocsr()
    char_eval = lemma_char_vectorizer.transform(plain[n_train:]).tocsr()

    x_train = sparse.hstack([word_train, char_train], format="csr")
    x_eval = sparse.hstack([word_eval, char_eval], format="csr")

    model = OneVsRestClassifier(
        LogisticRegression(
            C=2.5,
            class_weight="balanced",
            solver="liblinear",
            max_iter=2000,
            random_state=SEED,
        ),
        n_jobs=1,
    )
    model.fit(x_train, y)
    logistic = model.predict_proba(x_eval).astype(np.float32)

    knn_20 = knn_scores(similarity, y, k=20, power=1.5, gamma=0.1)
    knn_120 = knn_scores(similarity, y, k=120, power=3.0, gamma=0.3)

    base = blend([
        (logistic, 1.0),
        (knn_20, 1.0),
        (knn_120, 0.05),
    ])

    cooccurrence = y.T @ y
    frequencies = y.sum(axis=0)
    np.fill_diagonal(cooccurrence, 0.0)
    dependency = np.log1p(
        cooccurrence * len(y)
        / np.maximum(frequencies[:, None] * frequencies[None, :], 1e-8)
    ) * (cooccurrence > 0)
    np.fill_diagonal(dependency, 0.0)

    scaled = row_scale(base)
    top5 = np.empty((len(base), 5), dtype=np.int32)
    columns = np.arange(base.shape[1])
    for i, row in enumerate(scaled):
        top5[i] = np.lexsort((columns, -row))[:5]

    seed_scores = np.zeros_like(scaled, dtype=np.float32)
    rows = np.arange(len(seed_scores))[:, None]
    seed_scores[rows, top5] = scaled[rows, top5]
    propagated = seed_scores @ dependency.astype(np.float32)

    return row_scale(base) + 0.35 * row_scale(propagated)

In [ ]:
def bm25(
    document_counts: sparse.csr_matrix,
    query_counts: sparse.csr_matrix,
    k1: float,
    b: float,
) -> np.ndarray:
    document_counts = document_counts.tocsr().astype(np.float32)
    query_counts = query_counts.tocsr()
    n_documents = document_counts.shape[0]

    document_frequency = np.asarray((document_counts > 0).sum(axis=0)).ravel()
    idf = np.log(
        1.0 + (n_documents - document_frequency + 0.5) / (document_frequency + 0.5)
    ).astype(np.float32)

    lengths = np.asarray(document_counts.sum(axis=1)).ravel().astype(np.float32)
    average_length = max(float(lengths.mean()), 1e-8)

    weighted = document_counts.copy()
    rows = np.repeat(np.arange(n_documents), np.diff(weighted.indptr))
    term_frequency = weighted.data.copy()
    norm = k1 * (1.0 - b + b * lengths[rows] / average_length)
    weighted.data = (
        term_frequency * (k1 + 1.0)
        / (term_frequency + norm)
        * idf[weighted.indices]
    )

    return ((query_counts > 0).astype(np.float32) @ weighted.T).toarray().astype(np.float32)


def make_chunks(
    title: str,
    structure: str,
    body: str,
    window: int = 180,
    stride: int = 120,
) -> list[str]:
    words = body.split()
    prefix = f"{title} {structure}".strip()

    if not words:
        return [prefix]
    if len(words) <= window:
        return [f"{prefix} {body}".strip()]

    chunks = []
    for start in range(0, len(words), stride):
        part = words[start:start + window]
        if not part:
            break
        chunks.append(f"{prefix} {' '.join(part)}".strip())
        if start + window >= len(words):
            break

    return chunks


def article_scores(
    articles: pd.DataFrame,
    eval_queries: pd.DataFrame,
    normalizer: TextNormalizer,
) -> np.ndarray:
    bodies, structures = zip(*(extract_article_text(value) for value in articles["body"]))

    titles_lemma = [normalizer.normalize(value, lemmatize=True) for value in articles["title"]]
    structures_lemma = [normalizer.normalize(value, lemmatize=True) for value in structures]
    bodies_lemma = [normalizer.normalize(value, lemmatize=True) for value in bodies]
    full_lemma = [
        f"{title} {structure} {body}".strip()
        for title, structure, body in zip(titles_lemma, structures_lemma, bodies_lemma)
    ]

    title_plain = [normalize_basic(value, remove_stopwords=False) for value in articles["title"]]
    structure_plain = [normalize_basic(value, remove_stopwords=False) for value in structures]
    rich_plain = [
        f"{title} {structure}".strip()
        for title, structure in zip(title_plain, structure_plain)
    ]

    query_lemma = [normalizer.normalize(value, lemmatize=True) for value in eval_queries["query_text"]]
    query_plain = [normalizer.normalize(value, lemmatize=False) for value in eval_queries["query_text"]]

    count_vectorizer = CountVectorizer(dtype=np.float32, ngram_range=(1, 1), min_df=1)
    document_counts = count_vectorizer.fit_transform(full_lemma).tocsr()
    query_counts = count_vectorizer.transform(query_lemma).tocsr()
    full_bm25 = bm25(document_counts, query_counts, k1=4.0, b=0.9)

    tfidf_vectorizer = TfidfVectorizer(
        dtype=np.float32,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=1,
    )
    document_tfidf = tfidf_vectorizer.fit_transform(full_lemma)
    query_tfidf = tfidf_vectorizer.transform(query_lemma)
    full_tfidf = (query_tfidf @ document_tfidf.T).toarray().astype(np.float32)

    char_vectorizer = TfidfVectorizer(
        dtype=np.float32,
        analyzer="char_wb",
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=1,
    )
    structure_tfidf = char_vectorizer.fit_transform(rich_plain)
    query_char = char_vectorizer.transform(query_plain)
    structure_char = (query_char @ structure_tfidf.T).toarray().astype(np.float32)

    chunks = []
    chunk_article_columns = []
    for article_column, (title, structure, body) in enumerate(
        zip(titles_lemma, structures_lemma, bodies_lemma)
    ):
        article_chunks = make_chunks(title, structure, body)
        chunks.extend(article_chunks)
        chunk_article_columns.extend([article_column] * len(article_chunks))

    chunk_vectorizer = CountVectorizer(dtype=np.float32, ngram_range=(1, 1), min_df=1)
    chunk_counts = chunk_vectorizer.fit_transform(chunks).tocsr()
    chunk_query_counts = chunk_vectorizer.transform(query_lemma).tocsr()
    chunk_raw = bm25(chunk_counts, chunk_query_counts, k1=2.0, b=0.45)

    chunk_max = np.zeros((len(eval_queries), len(articles)), dtype=np.float32)
    chunk_article_columns = np.asarray(chunk_article_columns, dtype=np.int32)
    for article_column in range(len(articles)):
        indices = np.flatnonzero(chunk_article_columns == article_column)
        chunk_max[:, article_column] = chunk_raw[:, indices].max(axis=1)

    return blend([
        (full_bm25, 1.0),
        (chunk_max, 0.8),
        (structure_char, 0.3),
        (full_tfidf, 0.3),
    ])

## 5. Обучение и формирование `answer.csv`

In [ ]:
def build_target_matrix(data: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    targets = [
        list(dict.fromkeys(map(int, value.split())))
        for value in data["ground_truth"].astype(str)
    ]
    label_ids = np.array(
        sorted({article_id for row in targets for article_id in row}),
        dtype=np.int64,
    )
    label_to_column = {
        article_id: column for column, article_id in enumerate(label_ids)
    }

    y = np.zeros((len(data), len(label_ids)), dtype=np.float32)
    for row, target_ids in enumerate(targets):
        for article_id in target_ids:
            y[row, label_to_column[article_id]] = 1.0

    return y, label_ids

y, label_ids = build_target_matrix(calibration)

In [ ]:
normalizer = TextNormalizer()

supervised = supervised_scores(calibration, test, y, normalizer)

articles_retrieval = article_scores(articles, test, normalizer)

In [ ]:
article_ids = articles["article_id"].to_numpy(dtype=np.int64)
article_to_column = {
    article_id: column for column, article_id in enumerate(article_ids)
}
known_columns = np.array(
    [article_to_column[article_id] for article_id in label_ids],
    dtype=np.int32,
)

supervised_full = np.zeros((len(test), len(article_ids)), dtype=np.float32)
supervised_full[:, known_columns] = supervised

article_adjusted = row_scale(articles_retrieval)
unknown_columns = np.ones(len(article_ids), dtype=bool)
unknown_columns[known_columns] = False
article_adjusted[:, unknown_columns] *= UNKNOWN_ARTICLE_WEIGHT

final_scores = row_scale(supervised_full) + ARTICLE_CHANNEL_WEIGHT * article_adjusted

predictions = []
for row in final_scores:
    order = top_k_indices(row, article_ids, TOP_K)
    predictions.append(" ".join(map(str, article_ids[order])))

answer = pd.DataFrame({
    "query_id": test["query_id"],
    "answer": predictions,
})

In [ ]:
valid_article_ids = set(article_ids.tolist())

assert list(answer.columns) == ["query_id", "answer"]
assert len(answer) == len(test)
assert answer["query_id"].tolist() == test["query_id"].tolist()
assert answer["query_id"].is_unique

for query_id, value in answer.itertuples(index=False):
    ids = list(map(int, value.split()))
    assert 1 <= len(ids) <= TOP_K, f"Некорректный top-k для query_id={query_id}"
    assert len(ids) == len(set(ids)), f"Повтор article_id для query_id={query_id}"
    assert set(ids).issubset(valid_article_ids), f"Неизвестный article_id для query_id={query_id}"

answer.to_csv(OUTPUT_FILE, index=False, lineterminator="\r\n")

print(f"Файл сохранён: {OUTPUT_FILE.resolve()}")
display(answer.head())